# Segmentation with Ultralytics YOLO Model

This notebook provides a step-by-step guide for utilizing the Ultralytics YOLO segmentation model. It outlines the process of building an `AI Inference Server` (AI IS) pipeline, including instructions for integrating Ultralytics while substituting its `opencv-python` dependency, which is not supported on AI IS, with `opencv-python-headless`. __This feature is only available on AI SDK 2.6.0 or higher, and AI IS version 2.6.0 or higher.__

The Ultralytics YOLO segmentation model enables precise localization of objects within images. It features detection and classification capabilities with pixel-level mask prediction. In this tutorial, we use the model to detect objects and calculate their size relative to the size of the image.

## Loading an input 

First, let's take a look at an input image we want to use for segmentation. We will use the `cv2.imread` function to load the image, which returns a numpy array in BGR format. The image shape is `height x width x 3`, and the data type is `uint8`. Let's visualize the input image to confirm it has loaded correctly.

In [ ]:
import cv2
from matplotlib import pyplot as plt

img = cv2.imread("../images/bus.jpg")  # img is in BGR format

plt.axis("off")
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))  # imshow expects RGB format

print(f"Image shape: {img.shape}")
print(f"Image type: {img.dtype}")

## Loading the model

We load the model using the `ultralytics` Python package. The following code will download the [YOLO v11 nano segmentation model](https://docs.ultralytics.com/models/yolo11/) to the `../models/` folder.

In [ ]:
from ultralytics import YOLO

model = YOLO(f"../models/yolo11n-seg.pt")

## Running the model for inference

We can easily run the model by feeding it with the image. In case of a numpy array input, [the model expects](https://docs.ultralytics.com/modes/predict/#inference-sources) a BGR image of size `height x width x channel` of `uint8` type. This means that in our case we don't have to modify the input. The model also takes care of the preprocessing and postprocessing steps, including image rescale, normalization, annotation, and so on.

The return value is a list with a single element in our case.

In [ ]:
result = model(img)
result = result[0]

Note that the model actually rescaled our image to 640 x 480 (depicted as `width x height`).

### CPU vs GPU

Ultralytics can run inference on GPU when CUDA is available, or on CPU otherwise.

- If you call `model(img)` without a `device` argument, Ultralytics automatically picks a device.
- To force CPU, use `device="cpu"`.
- To force GPU, use `device="cuda:0"` (or `device=0`).

Before forcing GPU, we can confirm that both PyTorch and CUDA are available:

In [ ]:

import torch

print(f"PyTorch version: {torch.__version__}")
print(f"Built with CUDA: {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU count: {torch.cuda.device_count()}")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")

If `torch` is installed as `torch+cu128` or similar, it installs the various `nvidia-cuda` libraries as dependencies. We can also check through Ultralytics if the GPU is available.

In [ ]:
import ultralytics
ultralytics.checks()

An example run where we force CPU usage:

In [ ]:
result = model(img, device="cpu")
result = result[0]

We can assign specific GPU devices as well. If they are not available, Ultralytics falls back to CPU usage automatically.

In [ ]:
result = model(img, device="cuda:0")
result = result[0]

## Evaluating the results

The output result of the segmentation contains great amount of collected information. We can ask for an annotated image, which provides a visualization of the segmentation results, displaying the original image with color-coded classes, labels, segmentation masks, bounding boxes, and confidence scores for each detected object.

In [ ]:
annotated_img = result.plot()

plt.axis("off")
plt.imshow(cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB))

We can query the amount of time in milliseconds each step required.

In [ ]:
result.speed

We can get a summary of the model's findings.

In [ ]:
detection_results = [
    {key: value for key, value in s.items() if key != 'segments'}  # exclude the individual segments from the summary
    for s in result.summary()
]

detection_results

We can ask the overall area of each object found by counting the number of pixels of their masks. Dividing it with the overall area of the image, we get the relative size of each detected object. Note that the calculation happens on the rescaled image.

In [ ]:
for mask in result.masks.data:
    mask_area = mask.sum().item()
    total_area = mask.numel()
    print(f"{mask_area / total_area * 100.0:.1f}%")


We can get the class IDs and names:

In [ ]:
for class_id in list(result.boxes.cls.int().tolist()):
    print(result.names[class_id])

Here is an example of how to combine the class names and their areas, and visualize the masks for each detected object.

In [ ]:
class_ids = result.boxes.cls.int().tolist()
masks = result.masks.data

for class_id, mask in zip(class_ids, masks):
    class_label = result.names[class_id]
    
    mask_area = mask.sum().item()
    total_area = mask.numel()
    relative_area = mask_area / total_area
    print(f"Detected {class_label} occupying {relative_area * 100.0:.1f}% of the image.")

    plt.imshow(mask.cpu().numpy(), cmap='gray')
    plt.title(f"Mask for {class_label}")
    plt.axis("off")
    plt.show()

Notebook [20-CreateInferenceWrapper](./20-CreateInferenceWrapper.ipynb) shows how to create a Python wrapper around the model.